<a href="https://colab.research.google.com/github/Sampavi01/Advanced_Time_Series_Forecasting/blob/time_series/DL_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Deep Learning - Transformer Model

In [1]:
# --- Core Libraries & Plotting ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import io

# --- Helper for Colab File Upload ---
from google.colab import files

# --- Deep Learning Framework (TensorFlow) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout, TimeDistributed, Conv1D, MaxPooling1D, Flatten, MultiHeadAttention, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- Scikit-Learn Tools ---
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from scipy.special import inv_boxcox

# --- Plotting Style ---
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
# ---  Load Data  ---
from google.colab import files
print("Please upload your 'featured_aep_data.csv' file")
uploaded = files.upload()
print("\n✅ File uploaded successfully!")

Please upload your 'featured_aep_data.csv' file


Saving featured_aep_data.csv to featured_aep_data.csv

✅ File uploaded successfully!


In [3]:
# Next, upload the parameters file.
print("\nNext, please upload your 'model_parameters.joblib' file.")
uploaded_params = files.upload()
print(f"\n✅ Uploaded '{list(uploaded_params.keys())}' successfully!")


Next, please upload your 'model_parameters.joblib' file.


Saving model_parameters.joblib to model_parameters.joblib

✅ Uploaded '['model_parameters.joblib']' successfully!


In [4]:
import io
# --- 2. Load the Uploaded Data and Parameters ---
# Load the DataFrame from the uploaded CSV
df_ml = pd.read_csv(io.BytesIO(uploaded['featured_aep_data.csv']), index_col='Datetime', parse_dates=True)
print("\nFeatured DataFrame successfully loaded.")
print("Shape of loaded data:", df_ml.shape)

params = joblib.load( 'model_parameters.joblib')


Featured DataFrame successfully loaded.
Shape of loaded data: (121247, 24)


In [5]:
# --- Unpack Parameters ---
lambda_boxcox = params['lambda_boxcox']
train_end_idx = params['train_end_idx']
val_end_idx = params['val_end_idx']
TARGET_TRANSFORMED = params['target_col_transformed']
TARGET_ORIGINAL = params['target_col_original']
FEATURES = params['feature_columns']

In [6]:
# ---  Recreate Splits ---
# The original dataset started on '2004-10-01 01:00:00'.
# We can calculate how many rows were dropped by comparing the start date
# of our new DataFrame to the original start date.

original_start_date = pd.to_datetime('2004-10-01 01:00:00')
actual_start_date = df_ml.index.min() # The first timestamp in our loaded data

# The difference in hours is the number of rows that were dropped
time_difference = actual_start_date - original_start_date
rows_dropped = int(time_difference.total_seconds() / 3600)

print(f"Calculated that {rows_dropped} rows were dropped by the feature engineering process.")

# Now, adjust the original split indices by this amount
adjusted_train_end = train_end_idx - rows_dropped
adjusted_val_end = val_end_idx - rows_dropped

# Use the adjusted indices to split the new df_ml DataFrame
train_df = df_ml.iloc[:adjusted_train_end]  # Renamed for clarity, like in your original code
val_df = df_ml.iloc[adjusted_train_end:adjusted_val_end]
test_df = df_ml.iloc[adjusted_val_end:]

Calculated that 49 rows were dropped by the feature engineering process.


In [7]:
# --- Scale the Data ---
# Neural networks require input features to be scaled, typically between 0 and 1.
# IMPORTANT: We fit the scaler ONLY on the training data to prevent data leakage.
scaler = MinMaxScaler()
# We scale both features and the target together for easier sequence creation.
train_scaled = scaler.fit_transform(train_df[FEATURES + [TARGET_TRANSFORMED]])
val_scaled = scaler.transform(val_df[FEATURES + [TARGET_TRANSFORMED]])
test_scaled = scaler.transform(test_df[FEATURES + [TARGET_TRANSFORMED]])

print("\n✅ Data is loaded, split, and scaled. Ready for sequence creation.")


✅ Data is loaded, split, and scaled. Ready for sequence creation.


### ** Data Preparation for Sequence Models**

In [8]:
def create_sequences(data, sequence_length, target_index):
    """Creates sequences of data for time series forecasting."""
    X, y = [], []
    for i in range(len(data) - sequence_length):
        # The input sequence is the window of past data
        X.append(data[i:(i + sequence_length), :])
        # The target is the value of the target variable at the end of the window
        y.append(data[i + sequence_length, target_index])
    return np.array(X), np.array(y)

# --- Define Hyperparameters ---
SEQUENCE_LENGTH = 24 * 7 # Look back at the last 7 days of hourly data (168 hours)
TARGET_INDEX = len(FEATURES) # The target is the last column in our scaled data array

# --- Create the sequences for training, validation, and testing ---
X_train_seq, y_train_seq = create_sequences(train_scaled, SEQUENCE_LENGTH, TARGET_INDEX)
X_val_seq, y_val_seq = create_sequences(val_scaled, SEQUENCE_LENGTH, TARGET_INDEX)
X_test_seq, y_test_seq = create_sequences(test_scaled, SEQUENCE_LENGTH, TARGET_INDEX)

print("\n✅ Setup Complete. Data is ready for the Transformer model.")
print(f"Training sequence shape: {X_train_seq.shape}")


✅ Setup Complete. Data is ready for the Transformer model.
Training sequence shape: (84674, 168, 22)


###  Building, Training, and Evaluating the Transformer Model

This architecture is composed of several key parts:
1.  **Input Embedding:** A Dense layer that projects our 22 input features into a higher-dimensional space, allowing the model to learn richer representations.
2.  **Positional Encoding:** We add a simple, learnable embedding that gives the model information about the position of each of the 168 timesteps.
3.  **Transformer Encoder Blocks:** The core of the model. Each block contains a Multi-Head Self-Attention layer and a small Feed-Forward network. This is where the model learns the relationships between different points in time.
4.  **Final Prediction Head:** A pooling layer and a set of Dense layers (an MLP) that produce the final regression output.

--- SimpleRNN Model Summary ---


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │         5,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,633 (22.00 KB)

 Trainable params: 5,633 (22.00 KB)

 Non-trainable params: 0 (0.00 B)


--- Training SimpleRNN Model ---
Epoch 1/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - loss: 0.0543 - val_loss: 6.1451e-04
Epoch 2/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - loss: 0.0032 - val_loss: 4.1275e-04
Epoch 3/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - loss: 0.0015 - val_loss: 8.4688e-04
Epoch 4/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 0.0010 - val_loss: 2.5076e-04
Epoch 5/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - loss: 6.9484e-04 - val_loss: 2.9350e-04
Epoch 6/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 5.6587e-04 - val_loss: 4.1377e-04
Epoch 7/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - loss: 4.7708e-04 - val_loss: 4.6387e-04
Epoch 8/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 4.6761e-04 - val_loss: 1.5645e-04
Epoch 9/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - loss: 3.7980e-04 - val_loss: 1.8427e-04
Epoch 10/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 16s 12ms/step - loss: 3.7291e-04 - val_loss: 0.0011
Epoch

In [14]:
from keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling1D,Input

# --- 1. Define the Transformer Architecture ---

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    """Creates a single block of a Transformer encoder."""
    # Multi-Head Self-Attention
    x = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs) # Self-attention: query, key, and value are all the same
    x = Dropout(dropout)(x)
    # Add & Norm (Residual Connection)
    x = LayerNormalization(epsilon=1e-6)(inputs + x)

    # Feed-Forward Network
    ff_x = Dense(ff_dim, activation="relu")(x)
    ff_x = Dropout(dropout)(ff_x)
    ff_x = Dense(inputs.shape[-1])(ff_x)
    # Add & Norm (Second Residual Connection)
    return LayerNormalization(epsilon=1e-6)(x + ff_x)


def build_robust_transformer(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    """Builds the full Transformer model."""
    inputs = Input(shape=input_shape)
    x = inputs

    # --- Input Embedding ---
    # Project the input features into a higher-dimensional space (e.g., 128)
    embedding_dim = 128
    x = Dense(embedding_dim)(x)

    # --- Positional Encoding ---
    # Create a simple learnable positional encoding
    positions = tf.range(start=0, limit=input_shape[0], delta=1)
    positional_embedding = keras.layers.Embedding(input_dim=input_shape[0], output_dim=embedding_dim)(positions)
    x += positional_embedding # Add the positional information to the feature embeddings

    # --- Transformer Encoder Blocks ---
    # Stack multiple encoder blocks
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    # --- Final Prediction Head ---
    # Aggregate the information from all timesteps
    x = GlobalAveragePooling1D(data_format="channels_last")(x)
    # MLP (Multi-Layer Perceptron) for final regression
    for dim in mlp_units:
        x = Dense(dim, activation="relu")(x)
        x = Dropout(mlp_dropout)(x)
    outputs = Dense(1)(x)

    return Model(inputs, outputs)

# --- 2. Build and Compile the Model ---
input_shape = (SEQUENCE_LENGTH, X_train_seq.shape[2])  # (168, 22)
model = build_robust_transformer(
    input_shape=input_shape,
    head_size=128,
    num_heads=4,
    ff_dim=128,
    num_transformer_blocks=3,
    mlp_units=[128, 64],
    mlp_dropout=0.4,
    dropout=0.25
)

# A lower learning rate is often crucial for Transformers
optimizer = keras.optimizers.Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer, loss="mean_squared_error")
print("--- Robust Transformer Model Summary ---")
model.summary()


# --- 3. Train the Model ---
early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True) # Increased patience
print("\n--- Training Robust Transformer Model ---")
history = model.fit(X_train_seq, y_train_seq,
                    epochs=100, # Allow more epochs for the complex model to converge
                    batch_size=64,
                    validation_data=(X_val_seq, y_val_seq),
                    callbacks=[early_stopping], verbose=1)


# --- 4. Evaluate the Model ---
def evaluate_model(model, X_test, y_test_orig, scaler, lambda_val, seq_len, target_idx):
    """Reusable evaluation function."""
    preds_scaled = model.predict(X_test)
    dummy_array = np.zeros((len(preds_scaled), scaler.n_features_in_))
    dummy_array[:, target_idx] = preds_scaled.ravel()
    preds_boxcox = scaler.inverse_transform(dummy_array)[:, target_idx]
    preds_orig = inv_boxcox(preds_boxcox, lambda_val)
    true_values = y_test_orig.iloc[seq_len:]
    rmse = np.sqrt(mean_squared_error(true_values, preds_orig))
    return rmse

rmse_result = evaluate_model(model, X_test_seq, test_df[TARGET_ORIGINAL], scaler, lambda_boxcox, SEQUENCE_LENGTH, TARGET_INDEX)
print(f"\n--- Final Robust Transformer Model Performance ---")
print(f"Test Set RMSE: {rmse_result:.2f} MW")

--- Robust Transformer Model Summary ---


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 168, 22)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 168, 128)  │      2,944 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 168, 128)  │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 168, 128)  │    263,808 │ add_7[0][0],      │
│ (MultiHeadAttentio… │                   │            │ add_7[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 168, 128)  │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 168, 128)  │          0 │ add_7[0][0],      │
│                     │                   │            │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 128)  │        256 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 168, 128)  │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 168, 128)  │          0 │ dense_11[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 168, 128)  │     16,512 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 168, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 128)  │        256 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 168, 128)  │    263,808 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 168, 128)  │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 168, 128)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 128)  │        256 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 168, 128)  │     16,512 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 168, 128)  │          0 │ dense_13[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 919,809 (3.51 MB)

 Trainable params: 919,809 (3.51 MB)

 Non-trainable params: 0 (0.00 B)


--- Training Robust Transformer Model ---
Epoch 1/100
 121/1324 ━━━━━━━━━━━━━━━━━━━━ 1:01:23 3s/step - loss: 0.2868

KeyboardInterrupt: 